=== Cross-Validation Summary ===

MAE: 8.8815,

RMSE: 13.0289,

 R²: 0.9030

Best Fold: 3


✅ Weighted Ensemble Test Results:


  MAE:  9.8038

  RMSE: 13.5145
  
  R²:   0.8893

🏆 EXCELLENT PERFORMANCE ACHIEVED!

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Pre-processing

In [ ]:
import numpy as np
import pandas as pd

from IPython.display import display, HTML
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio


import seaborn as sns
from importlib import reload
import matplotlib.pyplot as plt
import matplotlib
import warnings

# Configure Jupyter Notebook
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 500)
pd.set_option('display.expand_frame_repr', False)
# pd.set_option('max_colwidth', -1)
display(HTML("<style>div.output_scroll { height: 35em; }</style>"))

reload(plt)
%matplotlib inline
%config InlineBackend.figure_format ='retina'

warnings.filterwarnings('ignore')

# configure plotly graph objects
pio.renderers.default = 'iframe'
# pio.renderers.default = 'vscode'

pio.templates["ck_template"] = go.layout.Template(
    layout_colorway = px.colors.sequential.Viridis,
#     layout_hovermode = 'closest',
#     layout_hoverdistance = -1,
    layout_autosize=False,
    layout_width=800,
    layout_height=600,
    layout_font = dict(family="Calibri Light"),
    layout_title_font = dict(family="Calibri"),
    layout_hoverlabel_font = dict(family="Calibri Light"),
#     plot_bgcolor="white",
)

# pio.templates.default = 'seaborn+ck_template+gridon'
pio.templates.default = 'ck_template+gridon'
# pio.templates.default = 'seaborn+gridon'
# pio.templates

In [ ]:
index_names = ['engine', 'cycle']
setting_names = ['setting_1', 'setting_2', 'setting_3']
sensor_names=[ "(Fan inlet temperature) (◦R)",
"(LPC outlet temperature) (◦R)",
"(HPC outlet temperature) (◦R)",
"(LPT outlet temperature) (◦R)",
"(Fan inlet Pressure) (psia)",
"(bypass-duct pressure) (psia)",
"(HPC outlet pressure) (psia)",
"(Physical fan speed) (rpm)",
"(Physical core speed) (rpm)",
"(Engine pressure ratio(P50/P2)",
"(HPC outlet Static pressure) (psia)",
"(Ratio of fuel flow to Ps30) (pps/psia)",
"(Corrected fan speed) (rpm)",
"(Corrected core speed) (rpm)",
"(Bypass Ratio) ",
"(Burner fuel-air ratio)",
"(Bleed Enthalpy)",
"(Required fan speed)",
"(Required fan conversion speed)",
"(High-pressure turbines Cool air flow)",
"(Low-pressure turbines Cool air flow)" ]
col_names = index_names + setting_names + sensor_names

In [ ]:
df_train = pd.read_csv('drive/MyDrive/archive/CMaps/train_FD003.txt',sep=r'\s+',header=None,index_col=False,names=col_names)
df_test = pd.read_csv('drive/MyDrive/archive/CMaps/test_FD003.txt',sep=r'\s+',header=None,index_col=False,names=col_names)
df_test_RUL = pd.read_csv('drive/MyDrive/archive/CMaps/RUL_FD003.txt',sep=r'\s+',header=None,index_col=False,names=['RUL'])

In [ ]:
keep_features = [
    "(Fan inlet temperature) (◦R)",
    "(LPC outlet temperature) (◦R)",
    "(HPC outlet temperature) (◦R)",
    "(LPT outlet temperature) (◦R)",
    "(Fan inlet Pressure) (psia)",
    "(bypass-duct pressure) (psia)",
    "(HPC outlet pressure) (psia)",
    "(Physical fan speed) (rpm)",
    "(Physical core speed) (rpm)",
    "(Engine pressure ratio(P50/P2)",
    "(HPC outlet Static pressure) (psia)",
    "(Ratio of fuel flow to Ps30) (pps/psia)",
    "(Corrected fan speed) (rpm)",
    "(Corrected core speed) (rpm)",
    "(Bypass Ratio) ",
    "(Burner fuel-air ratio)",
    "(Bleed Enthalpy)",
    "(Required fan speed)",
    "(Required fan conversion speed)",
    "(High-pressure turbines Cool air flow)",
    "(Low-pressure turbines Cool air flow)",
    "setting_1",
    "setting_2",
    "setting_3"
]

In [ ]:
features = list(keep_features)

In [ ]:
# define the maximum life of each engine, as this could be used to obtain the RUL at each point in time of the engine's life
df_train_RUL = df_train.groupby(['engine']).agg({'cycle':'max'})
df_train_RUL.rename(columns={'cycle':'life'},inplace=True)
df_train_RUL.head()

,life
engine,
1,259
2,253
3,222
4,272
5,213


In [ ]:
df_train=df_train.merge(df_train_RUL,how='left',on=['engine'])

In [ ]:
df_train['RUL']=df_train['life']-df_train['cycle']
df_train.drop(['life'],axis=1,inplace=True)

# the RUL prediction is only useful nearer to the end of the engine's life, therefore we put an upper limit on the RUL
# this is a bit sneaky, since it supposes that the test set has RULs of less than this value, the closer you are
# to the true value, the more accurate the model will be
df_train['RUL'][df_train['RUL']>125]=125
df_train.head()

,engine,cycle,setting_1,setting_2,setting_3,(Fan inlet temperature) (◦R),(LPC outlet temperature) (◦R),(HPC outlet temperature) (◦R),(LPT outlet temperature) (◦R),(Fan inlet Pressure) (psia),(bypass-duct pressure) (psia),(HPC outlet pressure) (psia),(Physical fan speed) (rpm),(Physical core speed) (rpm),(Engine pressure ratio(P50/P2),(HPC outlet Static pressure) (psia),(Ratio of fuel flow to Ps30) (pps/psia),(Corrected fan speed) (rpm),(Corrected core speed) (rpm),(Bypass Ratio),(Burner fuel-air ratio),(Bleed Enthalpy),(Required fan speed),(Required fan conversion speed),(High-pressure turbines Cool air flow),(Low-pressure turbines Cool air flow),RUL
0,1,1,-0.0005,0.0004,100.0,518.67,642.36,1583.23,1396.84,14.62,21.61,553.97,2387.96,9062.17,1.3,47.30,522.31,2388.01,8145.32,8.4246,0.03,391,2388,100.0,39.11,23.3537,125
1,1,2,0.0008,-0.0003,100.0,518.67,642.50,1584.69,1396.89,14.62,21.61,554.55,2388.00,9061.78,1.3,47.23,522.42,2388.03,8152.85,8.4403,0.03,392,2388,100.0,38.99,23.4491,125
2,1,3,-0.0014,-0.0002,100.0,518.67,642.18,1582.35,1405.61,14.62,21.61,554.43,2388.03,9070.23,1.3,47.22,522.03,2388.00,8150.17,8.3901,0.03,391,2388,100.0,38.85,23.3669,125
3,1,4,-0.0020,0.0001,100.0,518.67,642.92,1585.61,1392.27,14.62,21.61,555.21,2388.00,9064.57,1.3,47.24,522.49,2388.08,8146.56,8.3878,0.03,392,2388,100.0,38.96,23.2951,125
4,1,5,0.0016,0.0000,100.0,518.67,641.68,1588.63,1397.65,14.62,21.61,554.74,2388.04,9076.14,1.3,47.15,522.58,2388.03,8147.80,8.3869,0.03,392,2388,100.0,39.14,23.4583,125


In [ ]:
def create_sequences(df, window_size, stride, engine_col='engine'):
    sequences, targets, engine_ids = [], [], []

    for engine_id in df[engine_col].unique():
        engine_data = df[df[engine_col] == engine_id].sort_values('cycle')
        feature_cols = [col for col in engine_data.columns if col not in ['engine', 'cycle', 'RUL']]

        values = engine_data[feature_cols].values
        rul_values = engine_data['RUL'].values

        for i in range(0, len(values) - window_size + 1, stride):
            sequences.append(values[i:i + window_size])
            targets.append(rul_values[i + window_size - 1])
            engine_ids.append(engine_id)

    return np.array(sequences), np.array(targets), np.array(engine_ids)



# Scale features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
feature_cols = [col for col in df_train.columns if col not in ['engine', 'cycle', 'RUL']]
df_train[feature_cols] = scaler.fit_transform(df_train[feature_cols])
df_test[feature_cols] = scaler.transform(df_test[feature_cols])

In [ ]:
pip install tensorflow

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, regularizers
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# ============================================================
# ADVANCED RUL PREDICTION: Gated Ensemble BiLSTM + Attention (Optimized)
# ============================================================

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks, optimizers
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# MODIFIED create_sequences function to handle missing 'RUL' column for test set
def create_sequences(df, window_size, stride, engine_col='engine'):
    sequences, targets, engine_ids = [], [], []

    # Check if 'RUL' column exists in the DataFrame
    has_rul = 'RUL' in df.columns

    for engine_id in df[engine_col].unique():
        engine_data = df[df[engine_col] == engine_id].sort_values('cycle')
        feature_cols = [col for col in engine_data.columns if col not in ['engine', 'cycle', 'RUL']]

        values = engine_data[feature_cols].values

        if has_rul:
            rul_values = engine_data['RUL'].values
        else:
            # If RUL column is not present (e.g., for test data), create a dummy array.
            # These 'targets' will be discarded for X_test anyway.
            rul_values = np.array([])

        # Ensure there are enough data points for the window
        if len(values) < window_size:
            continue # Skip engines that are too short for a single sequence

        for i in range(0, len(values) - window_size + 1, stride):
            sequences.append(values[i:i + window_size])
            if has_rul: # Only append RUL target if RUL column exists
                targets.append(rul_values[i + window_size - 1])
            engine_ids.append(engine_id)

    return np.array(sequences), np.array(targets), np.array(engine_ids)


CONFIG = {
    'WINDOW_SIZE': 60,        # FD001 engines avg ~200 cycles; 30 is sharper
    'STRIDE': 1,
    'LSTM_UNITS': [128, 64],
    'DENSE_UNITS': 80,
    'DROPOUT_RATE': 0.2,      # Reduced — your CV→test gap suggests mild overfit
    'L2_REG': 0.0005,         # Slightly relaxed
    'BATCH_SIZE': 32,         # Smaller batch → better generalization
    'EPOCHS': 200,
    'LEARNING_RATE': 0.001,
    'PATIENCE': 20,           # Tighter early stopping
    'N_SPLITS': 5
}

print("\nOptimized Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")


X_train, y_train, train_engine_ids = create_sequences(
    df_train, CONFIG['WINDOW_SIZE'], CONFIG['STRIDE']
)
X_test, _, test_engine_ids = create_sequences(
    df_test, CONFIG['WINDOW_SIZE'], CONFIG['STRIDE']
)

print(f"Training sequences: {X_train.shape}")
print(f"Test sequences: {X_test.shape}")


# ============================================================
# Custom Layers
# ============================================================

class GatingLayer(layers.Layer):
    def __init__(self, num_branches, **kwargs):
        super(GatingLayer, self).__init__(**kwargs)
        self.num_branches = num_branches

    def build(self, input_shape):
        self.gate_weights = self.add_weight(
            shape=(input_shape[0][-1], self.num_branches),
            initializer='glorot_uniform',
            trainable=True
        )
        self.gate_bias = self.add_weight(
            shape=(self.num_branches,),
            initializer='zeros',
            trainable=True
        )

    def call(self, inputs):
        input_stats = tf.reduce_mean(inputs[0], axis=1)
        gate_scores = tf.nn.softmax(tf.matmul(input_stats, self.gate_weights) + self.gate_bias, axis=-1)
        gate_scores = tf.expand_dims(tf.expand_dims(gate_scores, 1), -1)
        stacked = tf.stack(inputs, axis=2)
        gated = stacked * gate_scores
        fused = tf.reduce_sum(gated, axis=2)
        return fused


class AttentionLayer(layers.Layer):
    """Attention mechanism to highlight critical timesteps after fusion."""
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(shape=(input_shape[-1], input_shape[-1]),
                                 initializer="glorot_uniform", trainable=True)
        self.b = self.add_weight(shape=(input_shape[-1],),
                                 initializer="zeros", trainable=True)
        self.u = self.add_weight(shape=(input_shape[-1],),
                                 initializer="glorot_uniform", trainable=True)

    def call(self, x):
        u_t = tf.tanh(tf.tensordot(x, self.W, axes=1) + self.b)
        attn_scores = tf.nn.softmax(tf.tensordot(u_t, self.u, axes=1), axis=1)
        attn_scores = tf.expand_dims(attn_scores, -1)
        context = tf.reduce_sum(x * attn_scores, axis=1)
        return context


# ============================================================
# Optimized Gated BiLSTM + Attention Model
# ============================================================

def build_gated_attention_bilstm(input_shape, config):
    inputs = layers.Input(shape=input_shape, name='input')

    # Parallel branches: LSTM, GRU, CNN
    branch_lstm = layers.LSTM(64, return_sequences=True,
                              kernel_regularizer=regularizers.l2(config['L2_REG']))(inputs)
    branch_lstm = layers.LayerNormalization()(branch_lstm)

    branch_gru = layers.GRU(64, return_sequences=True,
                            kernel_regularizer=regularizers.l2(config['L2_REG']))(inputs)
    branch_gru = layers.LayerNormalization()(branch_gru)

    branch_cnn = layers.Conv1D(64, 5, padding='same', activation='relu',
                               kernel_regularizer=regularizers.l2(config['L2_REG']))(inputs)
    branch_cnn = layers.Conv1D(64, 3, padding='same', activation='relu',
                               kernel_regularizer=regularizers.l2(config['L2_REG']))(branch_cnn)
    branch_cnn = layers.LayerNormalization()(branch_cnn)

    fused = GatingLayer(num_branches=3)([branch_lstm, branch_gru, branch_cnn])
    fused = layers.Dropout(config['DROPOUT_RATE'])(fused)

    # BiLSTM layers for sequential modeling
    x = layers.Bidirectional(
        layers.LSTM(config['LSTM_UNITS'][0],
                    return_sequences=True,
                    kernel_regularizer=regularizers.l2(config['L2_REG'])),
        name='bilstm_1'
    )(fused)
    x = layers.LayerNormalization()(x)
    x = layers.Dropout(config['DROPOUT_RATE'])(x)

    x = layers.Bidirectional(
        layers.LSTM(config['LSTM_UNITS'][1],
                    return_sequences=True,
                    kernel_regularizer=regularizers.l2(config['L2_REG'])),
        name='bilstm_2'
    )(x)
    x = layers.LayerNormalization()(x)
    x = layers.Dropout(config['DROPOUT_RATE'])(x)

    # Attention layer
    context = AttentionLayer()(x)

    # Dense head
    x = layers.Dense(config['DENSE_UNITS'], activation='relu')(context)
    x = layers.Dropout(config['DROPOUT_RATE'])(x)

    outputs = layers.Dense(1, activation='linear', name='rul_output')(x)

    model = models.Model(inputs, outputs, name='Gated_Attention_BiLSTM')

    opt = optimizers.Adam(learning_rate=config['LEARNING_RATE'])
    model.compile(
        optimizer=opt,
        loss=tf.keras.losses.Huber(delta=12.0),
        metrics=['mae', 'mse']
    )
    return model


# ============================================================
# Train with Advanced Callbacks
# ============================================================

# ============================================================
# Train with Advanced Callbacks
# ============================================================

def train_model_with_cv(X_train, y_train, engine_ids, input_shape, config, n_splits=5):
    from sklearn.model_selection import GroupKFold

    gkf = GroupKFold(n_splits=n_splits)
    results, models_list = [], []

    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, engine_ids), 1):
        print(f"\n{'='*25} FOLD {fold}/{n_splits} {'='*25}")
        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]

        model = build_gated_attention_bilstm(input_shape, config)

        cb = [
            callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.4, patience=6, min_lr=5e-6),
            callbacks.EarlyStopping(monitor='val_loss', patience=config['PATIENCE'],
                                    restore_best_weights=True),
            callbacks.ModelCheckpoint(f"best_fold_{fold}.h5", save_best_only=True)
        ]

        history = model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=config['EPOCHS'],
            batch_size=config['BATCH_SIZE'],
            verbose=1,
            callbacks=cb
        )

        val_loss, val_mae, val_mse = model.evaluate(X_val, y_val, verbose=0)
        val_rmse = np.sqrt(val_mse)

        y_pred_val = model.predict(X_val, verbose=0).flatten()
        val_r2 = r2_score(y_val, y_pred_val)

        results.append({'fold': fold, 'mae': val_mae, 'rmse': val_rmse, 'r2': val_r2})
        models_list.append(model)

        print(f"Fold {fold} Results -> MAE: {val_mae:.4f}, RMSE: {val_rmse:.4f}, R²: {val_r2:.4f}")

    print("\n=== Cross-Validation Summary ===")
    avg_mae  = np.mean([r['mae']  for r in results])
    avg_rmse = np.mean([r['rmse'] for r in results])
    avg_r2   = np.mean([r['r2']   for r in results])
    print(f"MAE: {avg_mae:.4f}, RMSE: {avg_rmse:.4f}, R²: {avg_r2:.4f}")

    best_model_idx = np.argmax([r['r2'] for r in results])
    print(f"\nBest Fold: {best_model_idx + 1}")

    return models_list, results   # ← returns ALL models now


# ============================================================
# Execute Training
# ============================================================

input_shape = (X_train.shape[1], X_train.shape[2])
all_models, cv_results = train_model_with_cv(
    X_train, y_train, train_engine_ids, input_shape, CONFIG, CONFIG['N_SPLITS']
)

# ============================================================
# Ensemble ALL folds for test prediction
# ============================================================

# ============================================================
# Weighted Ensemble by fold R² score
# ============================================================

fold_r2_scores = np.array([r['r2'] for r in cv_results])
fold_weights = fold_r2_scores / fold_r2_scores.sum()  # normalize to sum=1

print("\nFold weights:")
for i, (r2, w) in enumerate(zip(fold_r2_scores, fold_weights), 1):
    print(f"  Fold {i}: R²={r2:.4f}, weight={w:.4f}")

all_preds = []
for m in all_models:
    preds = m.predict(X_test, verbose=0).flatten()
    preds = np.clip(preds, 0, None)
    all_preds.append(preds)

# Weighted average
y_test_pred_all = np.average(all_preds, axis=0, weights=fold_weights)

unique_engines = sorted(set(test_engine_ids))
y_true, y_pred = [], []
for eng_id in unique_engines:
    mask = test_engine_ids == eng_id
    y_pred.append(y_test_pred_all[mask][-1])
    y_true.append(df_test_RUL.iloc[int(eng_id) - 1]['RUL'])

y_true, y_pred = np.array(y_true), np.array(y_pred)
test_mae  = mean_absolute_error(y_true, y_pred)
test_rmse = np.sqrt(mean_squared_error(y_true, y_pred))
test_r2   = r2_score(y_true, y_pred)

print(f"\n✅ Weighted Ensemble Test Results:")
print(f"  MAE:  {test_mae:.4f}")
print(f"  RMSE: {test_rmse:.4f}")
print(f"  R²:   {test_r2:.4f}")

if test_r2 >= 0.88:
    print("\n🏆 EXCELLENT PERFORMANCE ACHIEVED!")



Optimized Configuration:
  WINDOW_SIZE: 60
  STRIDE: 1
  LSTM_UNITS: [128, 64]
  DENSE_UNITS: 80
  DROPOUT_RATE: 0.2
  L2_REG: 0.0005
  BATCH_SIZE: 32
  EPOCHS: 200
  LEARNING_RATE: 0.001
  PATIENCE: 20
  N_SPLITS: 5
Training sequences: (18820, 60, 24)
Test sequences: (10762, 60, 24)

========================= FOLD 1/5 =========================
Epoch 1/200
471/471 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 357.3655 - mae: 34.9776 - mse: 2454.6763

471/471 ━━━━━━━━━━━━━━━━━━━━ 40s 37ms/step - loss: 357.0081 - mae: 34.9472 - mse: 2451.5505 - val_loss: 97.7982 - val_mae: 12.8026 - val_mse: 295.9217 - learning_rate: 0.0010
Epoch 2/200
470/471 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 85.1232 - mae: 11.5782 - mse: 247.3568

471/471 ━━━━━━━━━━━━━━━━━━━━ 36s 35ms/step - loss: 85.1047 - mae: 11.5764 - mse: 247.2785 - val_loss: 64.8602 - val_mae: 9.7423 - val_mse: 171.2328 - learning_rate: 0.0010
Epoch 3/200
470/471 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 72.6650 - mae: 10.3560 - mse: 199.7106

471/471 ━━━━━━━━━━━━━━━━━━━━ 16s 35ms/step - loss: 72.6569 - mae: 10.3553 - mse: 199.6709 - val_loss: 61.4425 - val_mae: 8.4395 - val_mse: 169.3261 - learning_rate: 0.0010
Epoch 4/200
471/471 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - loss: 63.3807 - mae: 9.4501 - mse: 166.6411

471/471 ━━━━━━━━━━━━━━━━━━━━ 18s 38ms/step - loss: 63.3775 - mae: 9.4497 - mse: 166.6293 - val_loss: 60.6282 - val_mae: 8.4686 - val_mse: 175.3146 - learning_rate: 0.0010
Epoch 5/200
471/471 ━━━━━━━━━━━━━━━━━━━━ 16s 34ms/step - loss: 57.9998 - mae: 8.9118 - mse: 146.8391 - val_loss: 64.5048 - val_mae: 9.4618 - val_mse: 176.1947 - learning_rate: 0.0010
Epoch 6/200
471/471 ━━━━━━━━━━━━━━━━━━━━ 23s 40ms/step - loss: 53.0152 - mae: 8.4442 - mse: 131.1328 - val_loss: 65.7432 - val_mae: 9.5041 - val_mse: 175.0021 - learning_rate: 0.0010
Epoch 7/200
471/471 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 49.9827 - mae: 8.1420 - mse: 120.6619

...

471/471 ━━━━━━━━━━━━━━━━━━━━ 22s 38ms/step - loss: 61.8605 - mae: 9.3710 - mse: 158.5904 - val_loss: 75.8578 - val_mae: 10.5596 - val_mse: 218.2089 - learning_rate: 0.0010
Epoch 4/200
470/471 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 55.8170 - mae: 8.6930 - mse: 139.7225

471/471 ━━━━━━━━━━━━━━━━━━━━ 17s 35ms/step - loss: 55.8136 - mae: 8.6928 - mse: 139.7064 - val_loss: 71.5159 - val_mae: 9.8155 - val_mse: 205.1047 - learning_rate: 0.0010
Epoch 5/200
471/471 ━━━━━━━━━━━━━━━━━━━━ 16s 34ms/step - loss: 50.5199 - mae: 8.2044 - mse: 121.7424 - val_loss: 84.9492 - val_mae: 11.3399 - val_mse: 264.6909 - learning_rate: 0.0010
Epoch 6/200
471/471 ━━━━━━━━━━━━━━━━━━━━ 16s 34ms/step - loss: 44.3915 - mae: 7.5636 - mse: 101.9726 - val_loss: 80.7152 - val_mae: 10.9409 - val_mse: 236.8458 - learning_rate: 0.0010
Epoch 7/200
471/471 ━━━━━━━━━━━━━━━━━━━━ 16s 34ms/step - loss: 42.0114 - mae: 7.2918 - mse: 96.4737 - val_loss: 88.5004 - val_mae: 11.6055 - val_mse: 267.7611 - learning_rate: 0.0010
Epoch 8/200
471/471 ━━━━━━━━━━━━━━━━━━━━ 21s 34ms/step - loss: 39.2187 - mae: 7.0049 - mse: 87.6309 - val_loss: 81.8491 - val_mae: 10.7236 - val_mse: 244.6896 - learning_rate: 0.0010
Epoch 9/200
471/471 ━━━━━━━━━━━━━━━━━━━━ 16s 34ms/step - loss: 38.7389 - mae: 6.9365 - mse: 86.

471/471 ━━━━━━━━━━━━━━━━━━━━ 27s 37ms/step - loss: 345.5058 - mae: 33.9515 - mse: 2348.5237 - val_loss: 71.1088 - val_mae: 10.0457 - val_mse: 197.5639 - learning_rate: 0.0010
Epoch 2/200
471/471 ━━━━━━━━━━━━━━━━━━━━ 16s 34ms/step - loss: 74.7703 - mae: 10.6000 - mse: 200.3870 - val_loss: 79.5195 - val_mae: 11.1488 - val_mse: 234.2884 - learning_rate: 0.0010
Epoch 3/200
470/471 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 61.7394 - mae: 9.3946 - mse: 154.8344

471/471 ━━━━━━━━━━━━━━━━━━━━ 16s 34ms/step - loss: 61.7252 - mae: 9.3931 - mse: 154.7938 - val_loss: 70.2629 - val_mae: 9.5899 - val_mse: 201.2285 - learning_rate: 0.0010
Epoch 4/200
471/471 ━━━━━━━━━━━━━━━━━━━━ 16s 35ms/step - loss: 49.4253 - mae: 8.1006 - mse: 117.5633 - val_loss: 76.0935 - val_mae: 10.1638 - val_mse: 237.6235 - learning_rate: 0.0010
Epoch 5/200
471/471 ━━━━━━━━━━━━━━━━━━━━ 18s 38ms/step - loss: 43.2743 - mae: 7.4929 - mse: 99.2409 - val_loss: 74.2974 - val_mae: 10.0545 - val_mse: 220.7538 - learning_rate: 0.0010
Epoch 6/200
471/471 ━━━━━━━━━━━━━━━━━━━━ 16s 35ms/step - loss: 40.9998 - mae: 7.2011 - mse: 92.9171 - val_loss: 76.8037 - val_mae: 10.0226 - val_mse: 230.0850 - learning_rate: 0.0010
Epoch 7/200
470/471 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 38.2245 - mae: 6.9610 - mse: 84.3652

471/471 ━━━━━━━━━━━━━━━━━━━━ 16s 35ms/step - loss: 38.2260 - mae: 6.9611 - mse: 84.3709 - val_loss: 68.8359 - val_mae: 8.9847 - val_mse: 201.3752 - learning_rate: 0.0010
Epoch 8/200
471/471 ━━━━━━━━━━━━━━━━━━━━ 16s 35ms/step - loss: 37.7357 - mae: 6.8560 - mse: 83.7063 - val_loss: 83.3661 - val_mae: 10.5457 - val_mse: 256.0340 - learning_rate: 0.0010
Epoch 9/200
471/471 ━━━━━━━━━━━━━━━━━━━━ 21s 35ms/step - loss: 35.5693 - mae: 6.5674 - mse: 78.5836 - val_loss: 82.7513 - val_mae: 10.4297 - val_mse: 259.6301 - learning_rate: 0.0010
Epoch 10/200
471/471 ━━━━━━━━━━━━━━━━━━━━ 16s 34ms/step - loss: 35.0724 - mae: 6.5808 - mse: 76.6374 - val_loss: 86.1914 - val_mae: 10.8023 - val_mse: 273.4341 - learning_rate: 0.0010
Epoch 11/200
471/471 ━━━━━━━━━━━━━━━━━━━━ 22s 38ms/step - loss: 34.9686 - mae: 6.5387 - mse: 76.2339 - val_loss: 72.9887 - val_mae: 9.5229 - val_mse: 216.2301 - learning_rate: 0.0010
Epoch 12/200
471/471 ━━━━━━━━━━━━━━━━━━━━ 16s 35ms/step - loss: 34.5596 - mae: 6.5164 - mse: 74.9

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# Prep: define y_pred_final and supporting variables
# ============================================================

y_pred_final = y_pred  # weighted ensemble predictions (from evaluation step)
final_r2     = test_r2
sorted_idx   = np.argsort(y_true)

# Single best model predictions (for comparison plot)
best_fold_idx   = np.argmax(fold_r2_scores)
y_pred_single   = np.clip(all_models[best_fold_idx].predict(X_test, verbose=0).flatten(), 0, None)

# Per-engine last-window for single model
y_pred_single_eng = []
for eng_id in unique_engines:
    mask = test_engine_ids == eng_id
    y_pred_single_eng.append(y_pred_single[mask][-1])
y_pred_single_eng = np.array(y_pred_single_eng)

y_pred_ens = y_pred_final  # alias for clarity in plot 5

# ============================================================
# Plot 1 — True vs Predicted sorted by True RUL
# ============================================================

plt.figure(figsize=(14, 4))
plt.plot(y_true[sorted_idx],       label='True RUL',
         marker='o', ms=3, color='steelblue')
plt.plot(y_pred_final[sorted_idx], label='Predicted RUL',
         marker='x', ms=3, color='tomato')
plt.xlabel('Engine (sorted by True RUL)')
plt.ylabel('RUL')
plt.title('FD003 — True vs Predicted RUL (Weighted Ensemble, sorted by True RUL)')
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# Plot 2 — Scatter: Predicted vs True
# ============================================================

plt.figure(figsize=(6, 6))
plt.scatter(y_true, y_pred_final, alpha=0.5, s=15, color='steelblue')
mn = min(y_true.min(), y_pred_final.min())
mx = max(y_true.max(), y_pred_final.max())
plt.plot([mn, mx], [mn, mx], 'r--', label='Perfect prediction')
plt.xlabel('True RUL')
plt.ylabel('Predicted RUL')
plt.title(f'FD003 — Predicted vs True RUL  (R²={final_r2:.3f})')
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# Plot 3 — Residual Distribution
# ============================================================

residuals = y_pred_final - y_true
plt.figure(figsize=(8, 4))
plt.hist(residuals, bins=30, color='steelblue', edgecolor='white')
plt.axvline(0, color='red', linestyle='--', label='Zero bias')
plt.axvline(residuals.mean(), color='orange', linestyle='--',
            label=f'Mean bias: {residuals.mean():+.2f}')
plt.xlabel('Prediction Error (pred − true)')
plt.ylabel('Count')
plt.title('FD003 — Residual Distribution')
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# Plot 4 — CV Fold Performance
# ============================================================

fold_labels = [f"Fold {r['fold']}" for r in cv_results]
fold_maes   = [r['mae'] for r in cv_results]
fold_r2s    = [r['r2']  for r in cv_results]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(fold_labels, fold_maes, color='steelblue')
axes[0].set_title('FD001 — Validation MAE per Fold')
axes[0].set_ylabel('MAE')

axes[1].bar(fold_labels, fold_r2s, color='tomato')
axes[1].set_title('FD003 — Validation R² per Fold')
axes[1].set_ylabel('R²')
axes[1].axhline(0.80, color='green', linestyle='--', label='Target R²=0.80')
axes[1].legend()
plt.tight_layout()
plt.show()

# ============================================================
# Plot 5 — Single Best Fold vs Weighted Ensemble vs True RUL
# ============================================================

plt.figure(figsize=(14, 4))
plt.plot(y_true[sorted_idx],                  label='True RUL',
         color='steelblue', lw=1.5)
plt.plot(y_pred_single_eng[sorted_idx],        label=f'Best Fold (Fold {best_fold_idx+1})',
         color='orange', lw=1, alpha=0.7)
plt.plot(y_pred_ens[sorted_idx],               label='Weighted Ensemble',
         color='tomato', lw=1.5)
plt.xlabel('Engine (sorted by True RUL)')
plt.ylabel('RUL')
plt.title('FD003 — Best Single Fold vs Weighted Ensemble vs True RUL')
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# Plot 6 — Fold Weights (FD002 specific — shows ensemble contribution)
# ============================================================

plt.figure(figsize=(8, 4))
bars = plt.bar(fold_labels, fold_weights, color='mediumpurple', edgecolor='white')
plt.axhline(1 / len(fold_weights), color='gray', linestyle='--',
            label=f'Uniform weight = {1/len(fold_weights):.3f}')
for bar, w in zip(bars, fold_weights):
    plt.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.002,
             f'{w:.3f}', ha='center', va='bottom', fontsize=9)
plt.xlabel('Fold')
plt.ylabel('Weight')
plt.title('FD003 — Ensemble Fold Weights (R²-normalized)')
plt.legend()
plt.tight_layout()
plt.show()